In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import subprocess
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 300)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

SOURCE_RECOVERY_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "source_recovery"
    / "catching_confirmed"
)

SOURCE_RECOVERY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [2]:
repository_final = pd.read_csv(
    REPORTS_DIR
    / "catching_repository_final.csv"
)

source_candidates = pd.read_csv(
    REPORTS_DIR
    / "catching_source_candidates.csv"
)

catching_targets = pd.read_csv(
    INTERIM_DIR
    / "catching_targets.csv"
)

In [3]:
print(
    "Repository final:",
    repository_final.shape
)

print(
    "Source candidates:",
    source_candidates.shape
)

print(
    "Targets:",
    catching_targets.shape
)

Repository final: (30, 3)
Source candidates: (268, 17)
Targets: (30, 8)


In [4]:
confirmed = repository_final[
    repository_final[
        "repository_status"
    ]
    ==
    "CONFIRMED_STRONG"
].copy()

display(
    confirmed
)

print(
    "Confirmados fuertes:",
    len(confirmed)
)

assert len(confirmed) == 2

,site,repository_status,selected_repo
20,startbootstrap-3-col-portfolio,CONFIRMED_STRONG,StartBootstrap/startbootstrap-3-col-portfolio
26,startbootstrap-sb-admin-2,CONFIRMED_STRONG,StartBootstrap/startbootstrap-sb-admin-2


Confirmados fuertes: 2


In [5]:
confirmed_sources = (
    confirmed
    .merge(
        source_candidates[
            [
                "site",
                "repo_full_name",
                "clone_url",
                "html_url",
                "default_branch",
                "license_key",
                "license_name"
            ]
        ],
        left_on=[
            "site",
            "selected_repo"
        ],
        right_on=[
            "site",
            "repo_full_name"
        ],
        how="left",
        validate="one_to_one"
    )
)

In [6]:
display(
    confirmed_sources
)

assert (
    confirmed_sources[
        "clone_url"
    ].notna().all()
)

print(
    "Metadata completa ✅"
)

,site,repository_status,selected_repo,repo_full_name,clone_url,html_url,default_branch,license_key,license_name
0,startbootstrap-3-col-portfolio,CONFIRMED_STRONG,StartBootstrap/startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,https://github.com/StartBootstrap/startbootstrap-3-col-portfolio.git,https://github.com/StartBootstrap/startbootstrap-3-col-portfolio,master,mit,MIT License
1,startbootstrap-sb-admin-2,CONFIRMED_STRONG,StartBootstrap/startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,https://github.com/StartBootstrap/startbootstrap-sb-admin-2.git,https://github.com/StartBootstrap/startbootstrap-sb-admin-2,master,mit,MIT License


Metadata completa ✅


In [7]:
def safe_directory_name(
    value
):

    return (
        str(value)
        .replace("/", "__")
        .replace("\\", "__")
    )

In [8]:
def clone_repository(
    repo_full_name,
    clone_url
):

    repo_dir = (
        SOURCE_RECOVERY_DIR
        / safe_directory_name(
            repo_full_name
        )
    )

    result = {
        "repo_full_name":
            repo_full_name,

        "local_path":
            str(repo_dir),

        "clone_status":
            None,

        "error":
            None
    }

    if (
        repo_dir.exists()
        and
        (
            repo_dir
            / ".git"
        ).exists()
    ):

        result[
            "clone_status"
        ] = "ALREADY_EXISTS"

        return result

    if repo_dir.exists():

        result[
            "clone_status"
        ] = "PATH_EXISTS_NOT_GIT"

        result[
            "error"
        ] = (
            "El directorio existe pero "
            "no contiene .git."
        )

        return result

    process = subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            clone_url,
            str(repo_dir)
        ],
        capture_output=True,
        text=True
    )

    if process.returncode != 0:

        result[
            "clone_status"
        ] = "CLONE_ERROR"

        result[
            "error"
        ] = process.stderr[
            :1000
        ]

        return result

    result[
        "clone_status"
    ] = "CLONED"

    return result

In [9]:
clone_records = []

for _, row in confirmed_sources.iterrows():

    print(
        "Clonando:",
        row[
            "repo_full_name"
        ]
    )

    result = clone_repository(
        repo_full_name=
            row[
                "repo_full_name"
            ],

        clone_url=
            row[
                "clone_url"
            ]
    )

    clone_records.append(
        result
    )

clone_registry = pd.DataFrame(
    clone_records
)

display(
    clone_registry
)

Clonando: StartBootstrap/startbootstrap-3-col-portfolio
Clonando: StartBootstrap/startbootstrap-sb-admin-2


,repo_full_name,local_path,clone_status,error
0,StartBootstrap/startbootstrap-3-col-portfolio,/Users/uriel/Documents/machine_learning/data/public_v2/source_recovery/catching_confirmed/StartBootstrap__startbootstrap-3-col-portfolio,CLONED,None
1,StartBootstrap/startbootstrap-sb-admin-2,/Users/uriel/Documents/machine_learning/data/public_v2/source_recovery/catching_confirmed/StartBootstrap__startbootstrap-sb-admin-2,CLONED,None


In [10]:
def git_value(
    repo_dir,
    args
):

    process = subprocess.run(
        [
            "git",
            "-C",
            str(repo_dir)
        ]
        +
        args,
        capture_output=True,
        text=True
    )

    if process.returncode != 0:
        return None

    return process.stdout.strip()

In [11]:
git_metadata_records = []

for _, row in clone_registry.iterrows():

    repo_dir = Path(
        row[
            "local_path"
        ]
    )

    if not (
        repo_dir
        / ".git"
    ).exists():
        continue

    commit_sha = git_value(
        repo_dir,
        [
            "rev-parse",
            "HEAD"
        ]
    )

    remote_url = git_value(
        repo_dir,
        [
            "remote",
            "get-url",
            "origin"
        ]
    )

    branch = git_value(
        repo_dir,
        [
            "rev-parse",
            "--abbrev-ref",
            "HEAD"
        ]
    )

    commit_date = git_value(
        repo_dir,
        [
            "show",
            "-s",
            "--format=%cI",
            "HEAD"
        ]
    )

    git_metadata_records.append({
        "repo_full_name":
            row[
                "repo_full_name"
            ],

        "commit_sha":
            commit_sha,

        "branch":
            branch,

        "commit_date":
            commit_date,

        "remote_url":
            remote_url
    })

git_metadata = pd.DataFrame(
    git_metadata_records
)

display(
    git_metadata
)

,repo_full_name,commit_sha,branch,commit_date,remote_url
0,StartBootstrap/startbootstrap-3-col-portfolio,f544d6140b13133be8953037fe4fb09a54172ea9,master,2019-01-31T16:45:37-05:00,https://github.com/StartBootstrap/startbootstrap-3-col-portfolio.git
1,StartBootstrap/startbootstrap-sb-admin-2,f0309881ef82794a1bd6257cd321801bc38a0f3d,master,2021-04-28T21:31:52-04:00,https://github.com/StartBootstrap/startbootstrap-sb-admin-2.git


In [12]:
confirmed_registry = (
    confirmed_sources
    .merge(
        clone_registry,
        on="repo_full_name",
        how="left",
        validate="one_to_one"
    )
    .merge(
        git_metadata,
        on="repo_full_name",
        how="left",
        validate="one_to_one"
    )
)

In [13]:
display(
    confirmed_registry[
        [
            "site",
            "repo_full_name",
            "repository_status",
            "commit_sha",
            "branch",
            "commit_date",
            "license_name",
            "local_path"
        ]
    ]
)

,site,repo_full_name,repository_status,commit_sha,branch,commit_date,license_name,local_path
0,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,CONFIRMED_STRONG,f544d6140b13133be8953037fe4fb09a54172ea9,master,2019-01-31T16:45:37-05:00,MIT License,/Users/uriel/Documents/machine_learning/data/public_v2/source_recovery/catching_confirmed/StartBootstrap__startbootstrap-3-col-portfolio
1,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,CONFIRMED_STRONG,f0309881ef82794a1bd6257cd321801bc38a0f3d,master,2021-04-28T21:31:52-04:00,MIT License,/Users/uriel/Documents/machine_learning/data/public_v2/source_recovery/catching_confirmed/StartBootstrap__startbootstrap-sb-admin-2


In [14]:
SKIP_DIRS = {
    ".git",
    "node_modules",
    "vendor",
    "dist",
    "build",
    "coverage",
    ".next"
}

In [15]:
def iter_repository_files(
    repo_dir
):

    repo_dir = Path(
        repo_dir
    )

    for path in repo_dir.rglob(
        "*"
    ):

        if not path.is_file():
            continue

        relative = path.relative_to(
            repo_dir
        )

        if any(
            part in SKIP_DIRS
            for part in relative.parts
        ):
            continue

        yield path

In [16]:
inventory_records = []

for _, row in confirmed_registry.iterrows():

    repo_dir = Path(
        row[
            "local_path"
        ]
    )

    for path in iter_repository_files(
        repo_dir
    ):

        relative = path.relative_to(
            repo_dir
        )

        try:

            size_bytes = (
                path.stat()
                .st_size
            )

        except OSError:

            size_bytes = np.nan

        inventory_records.append({
            "site":
                row[
                    "site"
                ],

            "repo_full_name":
                row[
                    "repo_full_name"
                ],

            "relative_path":
                str(
                    relative
                ),

            "extension":
                path.suffix.lower(),

            "size_bytes":
                size_bytes
        })

repository_inventory = pd.DataFrame(
    inventory_records
)

In [17]:
print(
    "Archivos:",
    len(
        repository_inventory
    )
)

display(
    repository_inventory.head(
        100
    )
)

Archivos: 69


,site,repo_full_name,relative_path,extension,size_bytes
0,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,index.html,.html,6660
1,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,LICENSE,,1093
2,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,README.md,.md,4568
3,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,.gitignore,,12
4,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,package-lock.json,.json,194277
5,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,package.json,.json,1004
6,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,gulpfile.js,.js,970
7,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,.travis.yml,.yml,162
8,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,css/3-col-portfolio.css,.css,441
9,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,buttons.html,.html,32956


In [18]:
extension_summary = (
    repository_inventory
    .groupby(
        [
            "repo_full_name",
            "extension"
        ]
    )
    .size()
    .reset_index(
        name="file_count"
    )
    .sort_values(
        [
            "repo_full_name",
            "file_count"
        ],
        ascending=[
            True,
            False
        ]
    )
)

display(
    extension_summary
)

,repo_full_name,extension,file_count
0,StartBootstrap/startbootstrap-3-col-portfolio,,2
4,StartBootstrap/startbootstrap-3-col-portfolio,.json,2
1,StartBootstrap/startbootstrap-3-col-portfolio,.css,1
2,StartBootstrap/startbootstrap-3-col-portfolio,.html,1
3,StartBootstrap/startbootstrap-3-col-portfolio,.js,1
5,StartBootstrap/startbootstrap-3-col-portfolio,.md,1
6,StartBootstrap/startbootstrap-3-col-portfolio,.yml,1
13,StartBootstrap/startbootstrap-sb-admin-2,.scss,23
9,StartBootstrap/startbootstrap-sb-admin-2,.html,14
10,StartBootstrap/startbootstrap-sb-admin-2,.js,7


In [19]:
source_extensions = {
    ".html",
    ".htm",
    ".css",
    ".js",
    ".jsx",
    ".ts",
    ".tsx"
}

web_source_inventory = (
    repository_inventory[
        repository_inventory[
            "extension"
        ].isin(
            source_extensions
        )
    ]
    .copy()
)

In [20]:
print(
    "Archivos fuente web:",
    len(
        web_source_inventory
    )
)

print(
    web_source_inventory
    .groupby(
        "repo_full_name"
    )
    .size()
)

display(
    web_source_inventory[
        [
            "repo_full_name",
            "relative_path",
            "extension",
            "size_bytes"
        ]
    ]
    .head(
        200
    )
)

Archivos fuente web: 26
repo_full_name
StartBootstrap/startbootstrap-3-col-portfolio     3
StartBootstrap/startbootstrap-sb-admin-2         23
dtype: int64


,repo_full_name,relative_path,extension,size_bytes
0,StartBootstrap/startbootstrap-3-col-portfolio,index.html,.html,6660
6,StartBootstrap/startbootstrap-3-col-portfolio,gulpfile.js,.js,970
8,StartBootstrap/startbootstrap-3-col-portfolio,css/3-col-portfolio.css,.css,441
9,StartBootstrap/startbootstrap-sb-admin-2,buttons.html,.html,32956
10,StartBootstrap/startbootstrap-sb-admin-2,utilities-border.html,.html,26234
11,StartBootstrap/startbootstrap-sb-admin-2,index.html,.html,42782
13,StartBootstrap/startbootstrap-sb-admin-2,forgot-password.html,.html,3543
14,StartBootstrap/startbootstrap-sb-admin-2,utilities-color.html,.html,27291
15,StartBootstrap/startbootstrap-sb-admin-2,register.html,.html,4730
16,StartBootstrap/startbootstrap-sb-admin-2,404.html,.html,22113


In [21]:
html_inventory = (
    repository_inventory[
        repository_inventory[
            "extension"
        ].isin(
            {
                ".html",
                ".htm"
            }
        )
    ]
    .copy()
)

print(
    "HTML totales:",
    len(
        html_inventory
    )
)

print(
    html_inventory
    .groupby(
        "repo_full_name"
    )
    .size()
)

display(
    html_inventory
)

HTML totales: 15
repo_full_name
StartBootstrap/startbootstrap-3-col-portfolio     1
StartBootstrap/startbootstrap-sb-admin-2         14
dtype: int64


,site,repo_full_name,relative_path,extension,size_bytes
0,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,index.html,.html,6660
9,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,buttons.html,.html,32956
10,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,utilities-border.html,.html,26234
11,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,index.html,.html,42782
13,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,forgot-password.html,.html,3543
14,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,utilities-color.html,.html,27291
15,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,register.html,.html,4730
16,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,404.html,.html,22113
17,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,login.html,.html,4705
18,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,blank.html,.html,21756


In [22]:
technology_files = {
    "package.json",
    "package-lock.json",
    "yarn.lock",
    "vite.config.js",
    "vite.config.ts",
    "webpack.config.js",
    "gulpfile.js",
    "gulpfile.mjs",
    "composer.json"
}

In [23]:
technology_audit = (
    repository_inventory[
        repository_inventory[
            "relative_path"
        ]
        .apply(
            lambda x:
                Path(x).name
                in technology_files
        )
    ]
    .copy()
)

display(
    technology_audit
)

,site,repo_full_name,relative_path,extension,size_bytes
4,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,package-lock.json,.json,194277
5,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,package.json,.json,1004
6,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,gulpfile.js,.js,970
25,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,package-lock.json,.json,642222
26,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,package.json,.json,1493
28,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,gulpfile.js,.js,3784


In [24]:
catching_direct_registry = (
    confirmed_registry[
        [
            "site",
            "repo_full_name",
            "commit_sha",
            "branch",
            "license_name",
            "local_path"
        ]
    ]
    .merge(
        catching_targets,
        on="site",
        how="left",
        validate="one_to_one"
    )
)

In [25]:
print(
    catching_direct_registry.shape
)

display(
    catching_direct_registry
)

(2, 13)


,site,repo_full_name,commit_sha,branch,license_name,local_path,source_dataset,label_source,label_level,H1_violation,H4_violation,H5_violation,H6_violation
0,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,f544d6140b13133be8953037fe4fb09a54172ea9,master,MIT License,/Users/uriel/Documents/machine_learning/data/public_v2/source_recovery/catching_confirmed/StartBootstrap__startbootstrap-3-col-portfolio,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,1
1,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,f0309881ef82794a1bd6257cd321801bc38a0f3d,master,MIT License,/Users/uriel/Documents/machine_learning/data/public_v2/source_recovery/catching_confirmed/StartBootstrap__startbootstrap-sb-admin-2,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,1


In [26]:
catching_direct_registry[
    "source_dataset"
] = "Catching_UX_Flaws"

catching_direct_registry[
    "label_level"
] = "LLM_DIRECT"

catching_direct_registry[
    "repository_quality"
] = "CONFIRMED_STRONG"

catching_direct_registry[
    "primary_training"
] = False

catching_direct_registry[
    "intended_use"
] = "PIPELINE_VALIDATION"

In [27]:
confirmed_registry.to_csv(
    REPORTS_DIR
    / "catching_confirmed_repository_registry.csv",
    index=False
)

repository_inventory.to_csv(
    REPORTS_DIR
    / "catching_confirmed_repository_inventory.csv",
    index=False
)

extension_summary.to_csv(
    REPORTS_DIR
    / "catching_confirmed_extension_summary.csv",
    index=False
)

html_inventory.to_csv(
    REPORTS_DIR
    / "catching_confirmed_html_inventory.csv",
    index=False
)

catching_direct_registry.to_csv(
    INTERIM_DIR
    / "catching_direct_validation_registry.csv",
    index=False
)

In [28]:
expected = [
    REPORTS_DIR
    / "catching_confirmed_repository_registry.csv",

    REPORTS_DIR
    / "catching_confirmed_repository_inventory.csv",

    REPORTS_DIR
    / "catching_confirmed_extension_summary.csv",

    REPORTS_DIR
    / "catching_confirmed_html_inventory.csv",

    INTERIM_DIR
    / "catching_direct_validation_registry.csv"
]

for path in expected:

    print(
        path.name,
        "✅" if path.exists()
        else "❌"
    )

catching_confirmed_repository_registry.csv ✅
catching_confirmed_repository_inventory.csv ✅
catching_confirmed_extension_summary.csv ✅
catching_confirmed_html_inventory.csv ✅
catching_direct_validation_registry.csv ✅


In [29]:
target_cols = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

print("Diversidad de targets:")

print(
    catching_direct_registry[
        target_cols
    ].nunique(
        dropna=False
    )
)

print("\nVectores Y distintos:")

display(
    catching_direct_registry[
        target_cols
    ].drop_duplicates()
)

Diversidad de targets:
H1_violation    1
H4_violation    1
H5_violation    1
H6_violation    1
dtype: int64

Vectores Y distintos:


,H1_violation,H4_violation,H5_violation,H6_violation
0,1,1.0,1,1


In [30]:
catching_validation_limitation = pd.DataFrame([
    {
        "source_dataset":
            "Catching_UX_Flaws",

        "confirmed_instances":
            len(catching_direct_registry),

        "unique_target_vectors":
            catching_direct_registry[
                target_cols
            ]
            .drop_duplicates()
            .shape[0],

        "classification_validation":
            False,

        "pipeline_validation":
            True,

        "reason":
            "Las dos interfaces CONFIRMED_STRONG poseen el mismo vector "
            "H1=H4=H5=H6=1; no existe variación de clase."
    }
])

catching_validation_limitation

,source_dataset,confirmed_instances,unique_target_vectors,classification_validation,pipeline_validation,reason
0,Catching_UX_Flaws,2,1,False,True,Las dos interfaces CONFIRMED_STRONG poseen el mismo vector H1=H4=H5=H6=1; no existe variación de clase.


In [31]:
catching_validation_limitation.to_csv(
    REPORTS_DIR
    / "catching_direct_validation_limitation.csv",
    index=False
)